In [58]:
import pandas as pd
import os
import numpy as np
import glob

from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()


import warnings
warnings.filterwarnings("ignore")

In [75]:
import glob

files = glob.glob("C:/Users/visco/Downloads/penyisihan-datavidia-10/ISPU/*.csv")
dfs = [pd.read_csv(f) for f in files]
ispu = pd.concat(dfs, ignore_index=True)

In [76]:
ispu = ispu[[
 'tanggal',
 'stasiun',
 'pm_sepuluh',
 'pm_duakomalima',
 'sulfur_dioksida',
 'karbon_monoksida',
 'ozon',
 'nitrogen_dioksida',
 'kategori'
]]


In [ ]:
ispu.replace(['-','---'], np.nan, inplace=True)

num_cols = [
 'pm_sepuluh',
 'pm_duakomalima',
 'sulfur_dioksida',
 'karbon_monoksida',
 'ozon',
 'nitrogen_dioksida'
]

for c in num_cols:
    ispu[c] = pd.to_numeric(ispu[c], errors='coerce')
    ispu[c].fillna(ispu[c].median(), inplace=True)

ispu.dropna(subset=['kategori','stasiun'], inplace=True)


In [ ]:
ispu['tanggal'] = pd.to_datetime(ispu['tanggal'], errors='coerce')
ispu = ispu[ispu['tanggal'] >= '2018-01-01']


In [ ]:
le_station = LabelEncoder()
ispu['stasiun_enc'] = le_station.fit_transform(ispu['stasiun'])

le_target = LabelEncoder()
ispu['kategori_enc'] = le_target.fit_transform(ispu['kategori'])


In [ ]:
ispu = ispu.sort_values("tanggal")

ispu['target_enc'] = ispu['kategori_enc'].shift(-1)
ispu.dropna(inplace=True)


In [ ]:
features = [
 'pm_sepuluh',
 'pm_duakomalima',
 'sulfur_dioksida',
 'karbon_monoksida',
 'ozon',
 'nitrogen_dioksida',
 'stasiun_enc'
]

X = ispu[features]
y = ispu['target_enc']


In [66]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
split = int(len(ispu)*0.8)

X_train = X.iloc[:split]
X_val   = X.iloc[split:]
y_train = y.iloc[:split]
y_val   = y.iloc[split:]


ValueError: Found array with 0 sample(s) (shape=(0, 7)) while a minimum of 1 is required by RandomForestClassifier.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)


In [ ]:
pred = model.predict(X_val)

print("F1 Macro:", f1_score(y_val, pred, average='macro'))
print(classification_report(y_val, pred))


TypeError: '<=' not supported between instances of 'str' and 'int'

In [ ]:
last_row = ispu.iloc[-1]
current_X = last_row[features].to_frame().T


In [ ]:
future_dates = pd.date_range(
    start = last_row['tanggal'] + pd.Timedelta(days=1),
    end   = "2025-09-01"
)

predictions = []

for d in future_dates:
    p = model.predict(current_X)[0]
    predictions.append(p)


In [ ]:
pred_label = le_target.inverse_transform(predictions)